# Single-Cell RNA-seq Analysis with Scanpy
**Data:** 10x Genomics PBMC samples (two donors)  
**Pipeline:** Quality control → Doublet detection → Normalization → Dimensionality reduction → Clustering

---

## Overview

Single-cell RNA sequencing (scRNA-seq) measures gene expression in thousands of individual cells simultaneously, enabling researchers to identify distinct cell populations, states, and transitions that would be invisible in bulk RNA-seq (which only measures average expression across a whole tissue sample).

This notebook implements a complete, standard scRNA-seq analysis pipeline using **Scanpy**, the most widely used Python toolkit for single-cell analysis. The workflow follows the canonical structure used across the field: quality control, filtering, doublet detection, normalization, dimensionality reduction, and unsupervised clustering.

**Dataset:** peripheral blood mononuclear cells (PBMCs) from two donor samples, provided as part of the official [Scanpy tutorial dataset](https://scanpy.readthedocs.io/en/1.11.x/tutorials/basics/clustering.html), originally sourced from 10x Genomics.

**Pipeline steps:**
1. Load and concatenate multi-sample 10x Genomics data
2. Compute and visualize quality control (QC) metrics
3. Filter low-quality cells and genes
4. Detect and flag doublets (artifacts where two cells are captured as one)
5. Identify highly variable genes
6. Run PCA for dimensionality reduction
7. Build a neighbor graph and compute a UMAP embedding
8. Cluster cells using the Leiden algorithm
9. Visualize clusters alongside QC metrics to assess cluster quality

## 1. Setup

In [ ]:
!pip install scanpy -q
!pip install igraph -q
!pip install scikit-misc -q

In [ ]:
from __future__ import annotations

import anndata as ad
import pooch       # handles downloading and caching the example dataset
import scanpy as sc

sc.settings.set_figure_params(dpi=50, facecolor="white")

## 2. Load and Combine Multi-Sample Data

We load two 10x Genomics PBMC samples (`s1d1` and `s1d3`) and concatenate them into a single `AnnData` object — the standard data structure used throughout the scverse ecosystem (Scanpy, AnnData, etc.) for representing annotated single-cell data.

Combining multiple samples up front, with a `sample` label retained, allows us to check for and account for batch effects throughout the pipeline.

In [ ]:
# pooch handles downloading and caching the dataset from its DOI-hosted repository
EXAMPLE_DATA = pooch.create(
    path=pooch.os_cache("scverse_tutorials"),
    base_url="doi:10.6084/m9.figshare.22716739.v1/",
)
EXAMPLE_DATA.load_registry_from_doi()

In [ ]:
samples = {
    "s1d1": "s1d1_filtered_feature_bc_matrix.h5",
    "s1d3": "s1d3_filtered_feature_bc_matrix.h5",
}
adatas = {}

for sample_id, filename in samples.items():
    path = EXAMPLE_DATA.fetch(filename)
    sample_adata = sc.read_10x_h5(path)
    sample_adata.var_names_make_unique()   # avoid duplicate gene names within a sample
    adatas[sample_id] = sample_adata

adata = ad.concat(adatas, label="sample")
adata.obs_names_make_unique()              # avoid duplicate cell barcodes across samples

print(adata.obs["sample"].value_counts())
adata

## 3. Quality Control

Before any biological interpretation, we need to identify and remove low-quality cells — empty droplets, dying cells, or technical artifacts. Three categories of marker genes are flagged for QC purposes:

- **Mitochondrial genes (MT-)** — high mitochondrial RNA content typically indicates dying or stressed cells, since mitochondrial transcripts leak out as cell membranes break down
- **Ribosomal genes (RPS/RPL)** — useful for detecting cells with unusual ribosomal content
- **Hemoglobin genes (HB)** — useful for flagging contaminating red blood cells in PBMC samples, which are not nucleated and shouldn't appear in scRNA-seq data

In [ ]:
adata.var["mt"]   = adata.var_names.str.startswith("MT-")              # mitochondrial genes (human; "Mt-" for mouse)
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))      # ribosomal genes
adata.var["hb"]   = adata.var_names.str.contains("^HB[^(P)]")           # hemoglobin genes, excluding HBP1

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

print("QC metrics added to adata.obs:")
print([c for c in adata.obs.columns if 'count' in c or 'pct' in c])

### 3.1 Visualize QC metrics

Violin plots show the distribution of key QC metrics across all cells: number of genes detected per cell, total UMI counts per cell, and percentage of mitochondrial reads. Outlier cells (very low gene counts, or very high mitochondrial percentage) are candidates for filtering.

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# A scatter plot helps reveal the relationship between sequencing depth,
# gene detection, and mitochondrial content simultaneously
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

**What to look for:** cells with very low total counts and gene counts, paired with high mitochondrial percentage, are likely dying or low-quality cells and should be filtered out before downstream analysis.

## 4. Filtering Low-Quality Cells and Genes

Based on the QC distributions above, we apply minimum thresholds:
- **Cells** must express at least 100 genes (removes empty droplets / very low-quality cells)
- **Genes** must be detected in at least 3 cells (removes genes with essentially no signal across the dataset)

These thresholds are dataset-dependent — the values used here follow the standard Scanpy tutorial defaults, but should be tuned based on the QC plots above for any new dataset.

In [ ]:
print(f"Before filtering: {adata.n_obs:,} cells, {adata.n_vars:,} genes")

sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)

print(f"After filtering:  {adata.n_obs:,} cells, {adata.n_vars:,} genes")

## 5. Doublet Detection

A "doublet" occurs when two cells are accidentally captured together in a single droplet during library preparation, producing a combined expression profile that looks like one cell but is actually a technical artifact of two. Left unaddressed, doublets can masquerade as novel or "hybrid" cell types during clustering.

We use **Scrublet** (built into Scanpy as `sc.pp.scrublet`) to computationally identify likely doublets based on simulated doublet profiles. We run this per-sample (`batch_key="sample"`) since doublet rates can vary by sample/batch.

In [ ]:
sc.pp.scrublet(adata, batch_key="sample")

print("Doublet detection results:")
print(adata.obs["predicted_doublet"].value_counts())

## 6. Preserving Raw Counts and Identifying Highly Variable Genes

Before normalizing the data, we preserve the original raw counts in a separate layer — many downstream tools and differential expression methods expect access to raw counts, even after the main expression matrix has been transformed.

We then identify **highly variable genes (HVGs)** — genes whose expression varies substantially across cells. Restricting downstream analysis (PCA, clustering) to HVGs reduces computational cost and noise from genes that show little biological signal across the dataset.

In [ ]:
# Preserve raw counts before any transformation
adata.layers["counts"] = adata.X.copy()

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=10000,
    flavor="seurat_v3",      # works directly on raw counts, recommended for multi-sample data
    batch_key="sample"        # identifies HVGs per-batch then combines, reducing batch-driven bias
)

In [ ]:
sc.pl.highly_variable_genes(adata)

## 7. Principal Component Analysis

PCA reduces the dataset from thousands of genes down to a much smaller number of principal components that capture most of the meaningful variation, serving as the foundation for all downstream dimensionality reduction and clustering steps.

In [ ]:
sc.tl.pca(adata)

### 7.1 How many PCs carry meaningful signal?

The variance ratio plot shows how much variance each successive principal component explains. Typically, after an initial steep drop, the curve flattens out — components beyond this "elbow" contribute diminishing returns and are often dominated by noise.

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

### 7.2 Visualize cells in PCA space

Plotting cells along different pairs of principal components, colored by sample and by mitochondrial percentage, helps check for batch effects (do samples separate when they shouldn't?) and confirms that QC-flagged cells aren't driving the overall structure.

In [ ]:
sc.pl.pca(
    adata,
    color=["sample", "sample", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
)

## 8. Neighbor Graph and UMAP Embedding

PCA captures global structure, but it's still too high-dimensional to interpret visually. We build a **k-nearest-neighbor graph** in PCA space, then use **UMAP** (Uniform Manifold Approximation and Projection) to project this graph down to two dimensions for visualization, preserving local neighborhood structure as faithfully as possible.

In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color="sample",
    size=7,   # smaller point size reduces overplotting in dense regions
)

**What to check:** if the two samples overlap substantially in UMAP space rather than forming entirely separate clusters, that's a good sign — it suggests cell populations are driven by genuine biological cell type differences rather than technical batch effects between samples.

## 9. Clustering with the Leiden Algorithm

The **Leiden algorithm** is a graph-based community detection method that partitions the neighbor graph into discrete clusters — in practice, these clusters typically correspond to distinct cell types or cell states. It's the current standard clustering method in the single-cell field, having succeeded the earlier Louvain algorithm due to better guarantees on cluster connectivity.

In [ ]:
# Using the igraph implementation with a fixed number of iterations is
# significantly faster than the default, especially on larger datasets
sc.tl.leiden(adata, flavor="igraph", n_iterations=2)

print(f"Number of clusters identified: {adata.obs['leiden'].nunique()}")
print(adata.obs['leiden'].value_counts().sort_index())

In [ ]:
sc.pl.umap(adata, color=["leiden"])

### 9.1 Cross-checking clusters against QC metrics

It's good practice to overlay QC metrics and doublet predictions onto the UMAP/cluster visualization. If a cluster is driven primarily by high doublet scores or low gene counts rather than genuine biology, it's a signal that cluster may be a technical artifact rather than a real cell population.

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden", "predicted_doublet", "doublet_score"],
    wspace=0.5,   # increase horizontal space between panels for readability
    size=3,
)

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden", "log1p_total_counts", "pct_counts_mt", "log1p_n_genes_by_counts"],
    wspace=0.5,
    ncols=2,
)

**Interpreting this final figure:** clusters with unusually high mitochondrial percentage or unusually low total counts relative to other clusters may warrant closer inspection or exclusion before any biological interpretation (e.g., cell type annotation) is performed.

## 10. Summary

This notebook implemented a complete, standard single-cell RNA-seq analysis pipeline using Scanpy:

1. **Data loading** — combined two 10x Genomics PBMC samples into a single AnnData object
2. **Quality control** — computed and visualized mitochondrial, ribosomal, and hemoglobin gene percentages alongside count distributions
3. **Filtering** — removed low-quality cells and rarely-detected genes
4. **Doublet detection** — flagged likely technical doublets using Scrublet
5. **Feature selection** — identified the top 10,000 highly variable genes across both samples
6. **Dimensionality reduction** — ran PCA, then built a neighbor graph and computed a UMAP embedding
7. **Clustering** — applied the Leiden algorithm to identify discrete cell populations
8. **Quality assessment** — cross-checked clusters against QC metrics and doublet scores to assess biological vs. technical sources of cluster structure

**Key takeaways:**
- QC should happen before any biological interpretation — filtering decisions made early in the pipeline shape every downstream result
- Running doublet detection and highly variable gene selection per-sample (`batch_key`) helps prevent batch-driven artifacts from dominating multi-sample analyses
- UMAP and clustering results should always be cross-validated against QC metrics — a cluster that's just "low quality cells" is a very different finding than a genuine novel cell population
- This pipeline (QC → filter → doublet detection → HVG → PCA → neighbors → UMAP → Leiden) is the standard structure used across the vast majority of published single-cell RNA-seq analyses, making it a transferable foundation for any new scRNA-seq dataset

**Potential extensions:**
- Annotate clusters with cell type labels using canonical marker genes (e.g., CD3 for T cells, CD19 for B cells, in this PBMC dataset)
- Perform differential expression analysis between clusters to identify cluster-defining marker genes
- Apply batch correction methods (e.g., Harmony, BBKNN) if stronger batch effects were observed between samples